# Inference: MAP Queries

All inference in conin goes through a single function: `map_query`. Given a
model (optionally with constraints and evidence), it finds the most likely
state assignment. This notebook covers the available inference methods
across all model types.

Prerequisites: familiarity with model construction
([HMM](HMM_basics.ipynb), [BN](BN_basics.ipynb), [MN](MN_basics.ipynb),
[DBN](DBN_basics.ipynb)) and constraint definition
([constraints notebook](constraints.ipynb)).

In [ ]:
import pyomo.environ as pe

from conin.hidden_markov_model import HiddenMarkovModel, ConstrainedHiddenMarkovModel
from conin.bayesian_network import (
    DiscreteBayesianNetwork, DiscreteCPD, ConstrainedDiscreteBayesianNetwork,
)
from conin.markov_network import (
    DiscreteMarkovNetwork, DiscreteFactor, ConstrainedDiscreteMarkovNetwork,
)
from conin.dynamic_bayesian_network import (
    DynamicDiscreteBayesianNetwork, ConstrainedDynamicDiscreteBayesianNetwork,
)
from conin.inference import map_query
from conin.constraints import (
    OracleConstraint, pyomo_constraint_fn, toulbar2_constraint_fn,
)
from conin.constraints.oracle import (
    has_minimum_number_of_occurences_constraint,
    has_maximum_number_of_occurences_constraint,
)

## Setup

We build the same four models used in the
[constraints notebook](constraints.ipynb) and prepare evidence for each.

In [ ]:
# --- HMM ---
hmm = HiddenMarkovModel()
hmm.load_model(
    start_probs={"sunny": 0.6, "rainy": 0.4},
    transition_probs={
        ("sunny", "sunny"): 0.7, ("sunny", "rainy"): 0.3,
        ("rainy", "sunny"): 0.4, ("rainy", "rainy"): 0.6,
    },
    emission_probs={
        ("sunny", "walk"): 0.6, ("sunny", "shop"): 0.3, ("sunny", "clean"): 0.1,
        ("rainy", "walk"): 0.1, ("rainy", "shop"): 0.4, ("rainy", "clean"): 0.5,
    },
)
hmm.set_seed(7)
hmm_observed = hmm.generate_observed(10)

# --- BN ---
bn = DiscreteBayesianNetwork(
    states={
        "Pollution": ["low", "high"], "Smoker": ["yes", "no"],
        "Cancer": ["yes", "no"], "Xray": ["positive", "negative"],
        "Dyspnoea": ["yes", "no"],
    },
    cpds=[
        DiscreteCPD(node="Pollution", values={"low": 0.9, "high": 0.1}),
        DiscreteCPD(node="Smoker", values={"yes": 0.3, "no": 0.7}),
        DiscreteCPD(node="Cancer", parents=["Smoker", "Pollution"], values={
            ("yes", "low"): {"yes": 0.03, "no": 0.97},
            ("yes", "high"): {"yes": 0.05, "no": 0.95},
            ("no", "low"): {"yes": 0.001, "no": 0.999},
            ("no", "high"): {"yes": 0.02, "no": 0.98},
        }),
        DiscreteCPD(node="Xray", parents=["Cancer"], values={
            "yes": {"positive": 0.9, "negative": 0.1},
            "no": {"positive": 0.2, "negative": 0.8},
        }),
        DiscreteCPD(node="Dyspnoea", parents=["Cancer"], values={
            "yes": {"yes": 0.65, "no": 0.35},
            "no": {"yes": 0.3, "no": 0.7},
        }),
    ],
)

# --- MN ---
mn = DiscreteMarkovNetwork(
    states={"A": [0, 1, 2], "B": [0, 1, 2], "C": [0, 1, 2]},
    factors=[
        DiscreteFactor(nodes=["A"], values={0: 1, 1: 1, 2: 2}),
        DiscreteFactor(nodes=["B"], values={0: 1, 1: 1, 2: 3}),
        DiscreteFactor(nodes=["C"], values={0: 1, 1: 2, 2: 1}),
        DiscreteFactor(nodes=["A", "B"], values={(i, j): 1 for i in range(3) for j in range(3)}),
        DiscreteFactor(nodes=["B", "C"], values={(i, j): 1 for i in range(3) for j in range(3)}),
        DiscreteFactor(nodes=["A", "C"], values={(i, j): 1 for i in range(3) for j in range(3)}),
    ],
)

# --- DBN ---
dbn = DynamicDiscreteBayesianNetwork()
dbn.dynamic_states = {"A": [0, 1], "B": [0, 1]}
dbn.cpds = [
    DiscreteCPD(node=("A", 0), values=[0.9, 0.1]),
    DiscreteCPD(node=("B", dbn.t), parents=[("A", dbn.t)],
               values={0: [0.2, 0.8], 1: [0.9, 0.1]}),
    DiscreteCPD(node=("A", dbn.t), parents=[("A", dbn.t - 1)],
               values={0: [0.2, 0.8], 1: [0.9, 0.1]}),
]

print("HMM observed:", hmm_observed)

## The `map_query` interface

```python
result = map_query(pgm, method=..., evidence=..., **options)
```

- `pgm` — any model or constrained model
- `method` — `"viterbi"`, `"a_star"`, `"integer_program"`, `"toulbar2"`,
  or `"variable_elimination"`
- `evidence` — observed values (format depends on model type)
- `stop` — time horizon for DBNs (required)

The result is a `Munch` with:
- `.solution.states` — the MAP assignment (present in all methods)
- `.solutions` — list of all returned solutions
- `.termination_condition` — `"ok"` or an error/limit message

Additional fields vary: Viterbi/A\* provide `.solution.hidden` and
`.solution.log_likelihood`; the integer program provides
`.solution.log_factor_sum`.

## Viterbi (HMM only)

The classic Viterbi algorithm finds the single most likely hidden-state
sequence via dynamic programming. No extra dependencies.

In [ ]:
result_viterbi = map_query(hmm, method="viterbi", evidence=hmm_observed)

print("Hidden:     ", result_viterbi.solution.states)
print("Log-lik:    ", result_viterbi.solution.log_likelihood)
print("Termination:", result_viterbi.termination_condition)

assert result_viterbi.termination_condition == "ok"

## A\* (HMM only)

A\* search returns the same result as Viterbi for the single best solution.
Its main advantages are **k-best decoding** and support for **oracle
constraints**.

In [ ]:
result_astar = map_query(hmm, method="a_star", evidence=hmm_observed)

print("Hidden:        ", result_astar.solution.states)
print("Matches Viterbi:", result_astar.solution.states == result_viterbi.solution.states)

assert result_astar.solution.states == result_viterbi.solution.states

### k-best decoding

Pass `num_solutions` to retrieve the top-k paths, ranked by
log-likelihood:

In [ ]:
result_kbest = map_query(hmm, method="a_star", evidence=hmm_observed, num_solutions=5)

for i, sol in enumerate(result_kbest.solutions):
    print(f"  {i + 1}. {sol.states}  (log-lik: {sol.log_likelihood:.4f})")

assert len(result_kbest.solutions) == 5
assert result_kbest.solutions[0].states == result_viterbi.solution.states

### Constrained A\*

Oracle constraints restrict which hidden paths are feasible. All returned
solutions satisfy the constraints.

In [ ]:
rainy_lb = has_minimum_number_of_occurences_constraint(val="rainy", count=7)
rainy_ub = has_maximum_number_of_occurences_constraint(val="rainy", count=8)

chmm_oracle = ConstrainedHiddenMarkovModel(hmm=hmm, constraints=[rainy_lb, rainy_ub])
chmm_oracle.initialize_chmm()

result_c_astar = map_query(chmm_oracle, method="a_star", evidence=hmm_observed)

print("Unconstrained:", result_viterbi.solution.states)
print(f"  rainy count: {result_viterbi.solution.states.count('rainy')}")
print("Constrained:  ", result_c_astar.solution.states)
print(f"  rainy count: {result_c_astar.solution.states.count('rainy')}")

assert 7 <= result_c_astar.solution.states.count("rainy") <= 8

## Integer Program (all models)

The integer programming method formulates MAP decoding as a mixed-integer
program. For unconstrained models it uses an LP relaxation; for constrained
models it solves a full IP.

NOTE: Requires a MIP solver. We use HiGHS here (`appsi_highs`).

In [ ]:
# HMM
result_hmm_ip = map_query(
    hmm, method="integer_program", evidence=hmm_observed, solver="appsi_highs",
)
print("HMM:", result_hmm_ip.solution.states)

# BN (no evidence — finds the most probable joint assignment)
result_bn_ip = map_query(bn, method="integer_program", solver="appsi_highs")
print("BN: ", result_bn_ip.solution.states)

# MN
result_mn_ip = map_query(mn, method="integer_program", solver="appsi_highs")
print("MN: ", result_mn_ip.solution.states)

# DBN (stop defines the time horizon)
result_dbn_ip = map_query(
    dbn, method="integer_program", solver="appsi_highs", stop=2,
)
print("DBN:", result_dbn_ip.solution.states)

assert result_hmm_ip.solution.states == result_viterbi.solution.states

### Evidence

Evidence fixes certain variables to observed values. The MAP query then
finds the most likely assignment for the remaining variables.

- **HMM**: evidence is a list of observed emissions.
- **BN / MN**: evidence is a dict mapping node names to observed states.
- **DBN**: evidence is a dict mapping `(node, time)` tuples to observed
  states.

In [ ]:
# BN with evidence: given a positive Xray, what is the most likely state?
result_bn_ev = map_query(
    bn, method="integer_program", solver="appsi_highs",
    evidence={"Xray": "positive"},
)
print("BN (no evidence):   ", result_bn_ip.solution.states)
print("BN (Xray=positive): ", result_bn_ev.solution.states)

# MN with evidence: fix A=0
result_mn_ev = map_query(
    mn, method="integer_program", solver="appsi_highs",
    evidence={"A": 0},
)
print("MN (no evidence):   ", result_mn_ip.solution.states)
print("MN (A=0):           ", result_mn_ev.solution.states)

### Constrained integer program

Pyomo constraints pair with the integer program solver. The same constraint
can be verified against constrained A\* for HMMs.

In [ ]:
# HMM — same constraint as the A* example above
@pyomo_constraint_fn()
def hmm_rainy_pyomo(M, D):
    M.lb = pe.Constraint(expr=sum(M.V("H", t, "rainy") for t in D.hmm.T) >= 7)
    M.ub = pe.Constraint(expr=sum(M.V("H", t, "rainy") for t in D.hmm.T) <= 8)


chmm_pyomo = ConstrainedHiddenMarkovModel(hmm=hmm, constraints=[hmm_rainy_pyomo])
chmm_pyomo.initialize_chmm()

result_c_ip = map_query(
    chmm_pyomo, method="integer_program", evidence=hmm_observed, solver="appsi_highs",
)
print("Constrained A*:", result_c_astar.solution.states)
print("Constrained IP:", result_c_ip.solution.states)
print("Match:         ", result_c_astar.solution.states == result_c_ip.solution.states)

assert result_c_astar.solution.states == result_c_ip.solution.states

In [ ]:
# BN — Dyspnoea and Xray must differ
@pyomo_constraint_fn()
def bn_differ_pyomo(M):
    M.c = pe.ConstraintList()
    M.c.add(M.V("Dyspnoea", "yes") + M.V("Xray", "positive") <= 1)
    M.c.add(M.V("Dyspnoea", "no") + M.V("Xray", "negative") <= 1)


cbn = ConstrainedDiscreteBayesianNetwork(bn, constraints=[bn_differ_pyomo])

result_cbn = map_query(cbn, method="integer_program", solver="appsi_highs")
print("BN unconstrained:", result_bn_ip.solution.states)
print("BN constrained:  ", result_cbn.solution.states)

## Toulbar2 (all models)

The Toulbar2 method converts the model into a cost function network and
solves it with the Toulbar2 solver. For HMMs, the model is automatically
converted through the pipeline HMM → DBN → BN → UAI format → Toulbar2
CFN.

NOTE: Requires the `pytoulbar2` package.

In [ ]:
# HMM
result_hmm_tb2 = map_query(hmm, method="toulbar2", evidence=hmm_observed)
print("HMM:", result_hmm_tb2.solution.states)

# MN
result_mn_tb2 = map_query(mn, method="toulbar2")
print("MN: ", result_mn_tb2.solution.states)

# DBN
result_dbn_tb2 = map_query(dbn, method="toulbar2", stop=2)
print("DBN:", result_dbn_tb2.solution.states)

assert result_hmm_tb2.solution.states == result_viterbi.solution.states
assert result_mn_tb2.solution.states == result_mn_ip.solution.states

### Constrained Toulbar2

In [ ]:
# MN — all different
@toulbar2_constraint_fn()
def mn_alldiff_tb2(M):
    for s in [0, 1, 2]:
        M.AddGeneralizedLinearConstraint(
            [M.V("A", s), M.V("B", s), M.V("C", s)], "<=", 1
        )


cmn = ConstrainedDiscreteMarkovNetwork(mn, constraints=[mn_alldiff_tb2])

result_cmn_tb2 = map_query(cmn, method="toulbar2")
print("MN unconstrained:", result_mn_tb2.solution.states)
print("MN constrained:  ", result_cmn_tb2.solution.states)

## Variable Elimination (BN / MN)

Variable elimination is an exact inference algorithm that sums out
variables one at a time. It is available for Bayesian networks and Markov
networks and supports oracle constraints (with `nodes`).

NOTE: Requires the `pgmpy` package. Variable elimination is exact but can
be computationally expensive — its cost grows exponentially with the
treewidth of the model. For large or densely connected models, the integer
program or Toulbar2 methods are typically more practical.

In [ ]:
try:
    result_bn_ve = map_query(bn, method="variable_elimination")
    print("BN (VE):", result_bn_ve.solution.states)

    result_mn_ve = map_query(mn, method="variable_elimination")
    print("MN (VE):", result_mn_ve.solution.states)

    # With evidence
    result_bn_ve_ev = map_query(
        bn, method="variable_elimination", evidence={"Xray": "positive"},
    )
    print("BN (VE, Xray=positive):", result_bn_ve_ev.solution.states)

    assert result_bn_ve.solution.states == result_bn_ip.solution.states
    assert result_mn_ve.solution.states == result_mn_ip.solution.states
except ImportError:
    print("pgmpy is not installed — skipping variable elimination examples.")

## Summary

### Inference methods at a glance

| Method | `map_query` string | Model types | Constraint types | k-best | Extra deps |
| --- | --- | --- | --- | --- | --- |
| Viterbi | `"viterbi"` | HMM | (unconstrained) | no | none |
| A\* | `"a_star"` | HMM | oracle | yes | none |
| Integer program | `"integer_program"` | all | pyomo, algebraic | no | MIP solver |
| Toulbar2 | `"toulbar2"` | all | toulbar2, algebraic | no | `pytoulbar2` |
| Variable elimination | `"variable_elimination"` | BN, MN | oracle (with `nodes`) | no | `pgmpy` |

### Evidence formats

| Model | Evidence format | Example |
| --- | --- | --- |
| HMM | list of observations | `["walk", "shop", "clean"]` |
| BN / MN | dict of node → state | `{"Xray": "positive"}` |
| DBN | dict of (node, time) → state | `{("W", 0): "Sunny"}` |

### Result structure

All methods return a `Munch` with:

- `.solution.states` — the MAP assignment (present in all methods)
- `.solutions` — list of all returned solutions
- `.termination_condition` — `"ok"`, or an error/limit message

Additional fields vary by method:

- Viterbi / A\*: `.solution.hidden`, `.solution.log_likelihood`
- Integer program: `.solution.log_factor_sum`, `.solvetime`